# 1. Tesseract + Line segmentation

In [3]:
import gc  # Garbage Collector
import os
import re
import time
import traceback
import unicodedata
from collections import deque

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import pytesseract
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageFilter
from pytesseract import Output
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm
import imutils

# Set Tesseract to use multiple threads internally
os.environ["OMP_THREAD_LIMIT"] = "8"

In [4]:
# !pip install imutils

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for imutils: filename=imutils-0.5.4-py3-none-any.whl size=25869 sha256=6d253527ed3ce1878450a90a0f727c03ba056043bcc09539d209c2bce7bfac61
  Stored in directory: /Users/harshamarsha/Library/Caches/pip/wheels/5b/76/96/ad0c321506837bef578cf3008df3916c23018435a355d9f6b1
Successfully built imutils

[notice] A new release of pip is available: 23.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [94]:
# --- Helper Functions ---
def save_entry_to_file(entry_str, file_path="ocr_results.txt"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")


def find_header_separator_cv2(pil_img):
    
    # 1. Convert to OpenCV grayscale
    img = np.array(pil_img.convert("L"))
    h, w = img.shape

    # 2. Focus on the TOP 30% and binarize
    # Higher threshold catches faint/thin header lines
    end_y = int(h * 0.3)

    roi = img[:end_y, :]

    _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)

    # 3. Horizontal detection kernel
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))

    # 4. Keep only strong horizontal structures
    detected_lines = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2
    )

    # 5. Find strongest horizontal line
    row_sums = np.sum(detected_lines, axis=1)

    if np.max(row_sums) > 0:

        header_y = np.argmax(row_sums)

        # crop slightly BELOW the header line
        return header_y + 10

    return 0


def find_dynamic_split_point(pil_img, debug=False):
    """
    Detect center gutter between two text columns.
    Works for historical dictionary/newspaper scans.
    """

    gray = np.array(pil_img.convert("L"))
    h, w = gray.shape

    # -----------------------------------
    # 2. Binary image
    # text = 1
    # background = 0
    # -----------------------------------
    bw = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # -----------------------------------
    # 3. Ignore page margins
    # -----------------------------------
    top = int(h * 0.05)
    bottom = int(h * 0.95)

    content = bw[top:bottom, :]

    # -----------------------------------
    # 4. Vertical projection
    # Count text pixels per column
    # -----------------------------------
    text_density = np.sum(content, axis=0)

    # Smooth heavily
    text_density = cv2.GaussianBlur(
        text_density.astype(np.float32), (101, 1), 0
    ).flatten()

    # -----------------------------------
    # 5. Search only near center
    # -----------------------------------
    search_start = int(w * 0.35)
    search_end = int(w * 0.65)

    center_profile = text_density[search_start:search_end]

    # -----------------------------------
    # 6. Find deepest whitespace valley
    # -----------------------------------
    split_rel = np.argmin(center_profile)
    split_x = search_start + split_rel

    # -----------------------------------
    # 7. Confidence scoring
    # Compare valley depth to nearby peaks
    # -----------------------------------
    valley = center_profile[split_rel]

    left_peak = np.max(center_profile[:split_rel]) if split_rel > 0 else 0

    right_peak = (
        np.max(center_profile[split_rel:]) if split_rel < len(center_profile) else 0
    )

    peak = max(left_peak, right_peak)

    confidence = 1.0 - (valley / (peak + 1e-6))

    if debug:
        print(f"Split x = {split_x}")
        print(f"Valley density = {valley:.2f}")
        print(f"Peak density = {peak:.2f}")
        print(f"Confidence = {confidence:.3f}")

    # -----------------------------------
    # 8. Fallback
    # -----------------------------------
    if confidence < 0.2:
        split_x = w // 2

    return split_x

def deskew_pil_image(image, max_skew=1.0, step=0.2):
    """
    Detects and corrects minor text skews by finding the angle that
    maximizes the variance of the horizontal projection profile.
    """
    # 1. Downscale drastically for extreme speed (variance check works fine at low res)
    scale_factor = 4
    small_img = image.resize((image.width // scale_factor, image.height // scale_factor)).convert('L')
    
    # Convert to binary numpy array (text = 1, background = 0)
    # Assumes dark text on light background
    arr = np.array(small_img)
    binary = (arr < 127).astype(np.uint8)
    
    best_angle = 0.0
    max_variance = -1.0
    
    # 2. Test angles in the realistic scan tilt range (e.g., -5 to +5 degrees)
    angles = np.arange(-max_skew, max_skew + step, step)
    
    for angle in angles:
        # Rotate the binary array using PIL to check profile alignment
        # We use a white background for the check mask since it's inverted to 0
        rotated_check = small_img.rotate(angle, resample=0, expand=False) # 0 = NEAREST
        check_arr = np.array(rotated_check)
        bin_check = check_arr < 127
        
        # Sum horizontally along rows
        row_sums = np.sum(bin_check, axis=1)
        
        # Calculate variance of the row profile
        variance = np.var(row_sums)
        
        if variance > max_variance:
            max_variance = variance
            best_angle = angle

    # 3. Apply the correction to the original high-res image
    if abs(best_angle) > 0.1:
        # print(f"[INFO] Deskewing slight tilt: {best_angle:.2f} degrees")
        # PIL rotates counter-clockwise naturally.
        # If best_angle is positive, it means the text was tilted clockwise, 
        # so rotating by +best_angle counter-clockwise straightens it.
        fill_value = (255, 255, 255) if image.mode == "RGB" else 255
        return image.rotate(best_angle, expand=True, resample=3, fillcolor=fill_value) # 3 = BICUBIC

        
    return image

In [95]:
# Main OCR Pipeline
def ocr_pdf_lineseg_pipeline(
    pdf_path,
    image_dpi=600,
    languages="fra+latn",
    ocr_config="--oem 3 --psm 3",
    output_file="ocr_results.txt",
    export_pics=False,
    start_page = 1,
    end_page = None,
):

    max_pages = pdfinfo_from_path(pdf_path)["Pages"]

    # helper: write buffered entry
    def write_entry(entry, marker=None):
        if not entry:
            return
        s = " ".join(entry).strip()
        if s.endswith(("-", "–", "—")):
            s = s[:-1].strip()
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"{s}\n{marker}\n\n" if marker else f"{s}\n\n")

    open(output_file, "w", encoding="utf-8").close()  # create fresh file

    current_entry, held_marker = [], None
        
    for page_num in tqdm(
        range(start_page, max_pages + 1), desc="Processing Dictionary", unit="page"
    ):
        if end_page and page_num >= end_page:
            break
        try:
            images = convert_from_path(
                pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num
            )
            if not images:
                continue
            img = images[0]

            # correct minor tilts
            img = deskew_pil_image(img)
        
            # crop out top header
            header_y = find_header_separator_cv2(img)

            if header_y is not None:
                if header_y > 0:
                    img = img.crop((0, header_y, img.size[0], img.size[1]))

            # Split the page into two
            w, h = img.size
            mid = find_dynamic_split_point(img, debug=False)  # updated with de-skew

            # Define a pixel buffer to safely step away from the divider line
            # w * 0.008 is usually roughly 10-20 pixels depending on image resolution << ---------------
            line_buffer = int(w * 0.008)

            columns = [
                # Left Column: Crops from left margin up to just BEFORE the line
                img.crop((w * 0, h * 0, mid + line_buffer, h)),
                # Right Column: Crops from just AFTER the line to the right margin
                img.crop((mid, h * 0, w * 1, h)),
            ]

            for i, col_img in enumerate(columns):

                if export_pics:
                    col_img.save(f"p{page_num}_{i+1}.png")

                data = pytesseract.image_to_data(
                    col_img, lang=languages, config=ocr_config, output_type=Output.DICT
                )
                col_img.close()

                # reconstruct lines from OCR words
                lines, curr_text, curr_x = [], [], None
                for j in range(len(data["text"])):
                    if int(data["conf"][j]) > 0:
                        if int(data["word_num"][j]) == 1:
                            if curr_text:
                                lines.append({"x": curr_x, "text": " ".join(curr_text)})
                            curr_x, curr_text = data["left"][j], [data["text"][j]]
                        else:
                            curr_text.append(data["text"][j])
                if curr_text:
                    lines.append({"x": curr_x, "text": " ".join(curr_text)})
                if not lines:
                    continue

                marker = f"COLUMN-START-P{page_num}-C{i+1}"

                newline_offset = 10
                WINDOW = 5

                def entry_interrupted():
                    return current_entry and not current_entry[-1].rstrip().endswith(
                        (")", "]", "}", ".")
                    )

                held_marker = marker
                x_positions = [l["x"] for l in lines]

                for idx, line in enumerate(lines):
                    txt = line["text"].strip()
                    x = line["x"]

                    # LOCAL NEIGHBORHOOD (excluding current line)
                    neighbors = []

                    for j in range(
                        max(0, idx - WINDOW), min(len(lines), idx + WINDOW + 1)
                    ):
                        if j != idx:
                            nx = x_positions[j]
                            if nx is not None:
                                neighbors.append(nx)

                    # fallback protection
                    if neighbors:
                        local_indent = np.median(neighbors)
                    else:
                        local_indent = np.median(
                            [v for v in x_positions if v is not None]
                        )

                    # FIX: handle x=None by replacing with local median
                    if x is None:
                        print("Replacing x=None with local median:", line)
                        x = local_indent
                    # ENTRY DETECTION
                    is_new = x < (local_indent - newline_offset)
                    # FIRST ENTRY
                    if not current_entry:
                        current_entry = [txt]
                        continue

                    # NEW ENTRY
                    if is_new:
                        write_entry(current_entry, held_marker)
                        held_marker = None
                        current_entry = [txt]

                    # CONTINUATION
                    elif entry_interrupted():
                        current_entry[-1] = current_entry[-1].rstrip() + " " + txt
                    else:
                        current_entry.append(txt)
            img.close()
            del images, img
            gc.collect()

        except Exception as e:
            print(f"Skipping page {page_num} due to error: {e}")
            traceback.print_exc()
            continue

    write_entry(current_entry, held_marker)
    return f"Success! Results appended to {output_file}"

In [105]:
filename = "input_dict_files/kikuyu_working_file.pdf"  # main
output_file = "kikuyu_extracted_v2.txt"

start_time = time.perf_counter()
extracted_content = ocr_pdf_lineseg_pipeline(
    filename,
    image_dpi=400,
    languages="script/Latin",
    ocr_config=f"--oem 3 --psm 1 -c load_system_dawg=0 -c load_freq_dawg=0",
    output_file=output_file,
    export_pics = False,  # debug
    start_page = 1,    # debug
    end_page = None      # debug
)

end_time = time.perf_counter()

print(f"Time taken to OCR: {(end_time-start_time)/60:.2f} min")
extracted_content

Processing Dictionary:   0%|          | 0/562 [00:00<?, ?page/s]

[INFO] Deskewing slight tilt: 0.60 degrees
[INFO] Deskewing slight tilt: 0.60 degrees
[INFO] Deskewing slight tilt: 1.00 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 1.00 degrees
[INFO] Deskewing slight tilt: 1.00 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 1.00 degrees
[INFO] Deskewing slight tilt: 0.20 degrees
[INFO] Deskewing slight tilt: 0.60 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
Replacing x=None with local median: {'x': None, 'text': '2. cylindrical wooden barrel'}
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 0.80 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] Deskewing slight tilt: 0.60 degrees
[INFO] Deskewing slight tilt: 0.20 degrees
[INFO] Deskewing slight tilt: 0.80 degrees
[INFO] Deskewing slight tilt: 0.40 degrees
[INFO] De

'Success! Results appended to kikuyu_extracted_v2.txt'

# 2. Text Cleaning

In [244]:
print(
    sorted(
        __import__("collections")
        .Counter(open("kikuyu_extracted_full_cleaned.txt", encoding="utf-8").read())
        .items(),
        key=lambda x: x[1],
    )
)

[('ʻ', 1), ('¢', 1), ('‚', 1), ('\\', 1), ('¡', 1), ('Œ', 1), ('Ł', 1), ('©', 1), ('>', 1), ('Đ', 1), ('§', 1), ('€', 2), ('¥', 2), ('%', 2), ('”', 2), ('+', 2), ('$', 2), ('Þ', 3), ('ə', 3), ('«', 3), ('£', 3), ('#', 4), ('“', 5), ('„', 5), ('þ', 6), ('đ', 6), ('°', 6), ('»', 6), ('=', 7), ('_', 7), ('Q', 8), ('—', 10), ('ħ', 13), ('{', 15), ('`', 16), ('¿', 18), ('ł', 18), ('@', 19), ('*', 19), ('Ĩ', 23), ('Z', 38), ('"', 39), ('Y', 58), ('J', 59), ('F', 86), ('W', 110), ('X', 126), ('‘', 143), ('D', 148), ('}', 161), (':', 182), ('H', 215), ('B', 234), ('V', 273), ('7', 275), ('8', 293), ('G', 299), ('9', 324), ('Ũ', 350), ('|', 429), ('0', 447), ('5', 483), ('E', 635), ("'", 756), ('6', 790), ('q', 926), ('z', 1031), ('!', 1052), ('K', 1123), ('3', 1155), ('4', 1219), ('P', 1248), ('L', 1312), ('R', 1399), ('O', 1449), ('/', 1474), ('&', 1558), ('U', 1634), ('?', 1690), ('M', 1792), ('S', 1828), ('A', 1929), ('I', 2191), ('N', 2348), ('x', 2470), ('T', 2478), ('C', 2581), ('’', 329

In [97]:
def fix_dictionary_diacritics(text):
    # 1. Decompose text to isolate base characters from their diacritics
    # e.g., 'ā' becomes 'a' + '\u0304', 'īï' becomes 'i' + '\u0304' + 'i' + '\u0308'
    decomposed_text = unicodedata.normalize("NFD", text)

    # 2. Pattern 1: Catch contiguous 'i' or 'u' clusters with any combining diacritics
    kikuyu_vowel_pattern = r"(?i)(?:i[\u0300-\u036f]*)+|(?:u[\u0300-\u036f]*)+"

    def handle_kikuyu_vowels(match):
        cluster = match.group(0)
        # Check if this i/u cluster actually has an OCR diacritic noise symbol
        has_diacritic = any("\u0300" <= char <= "\u036f" for char in cluster)

        if has_diacritic:
            base_vowel = cluster[0] # we consider only the first vowel as the base char
            if base_vowel.lower() == "i":
                return "Ĩ" if base_vowel.isupper() else "ĩ"
            elif base_vowel.lower() == "u":
                return "Ũ" if base_vowel.isupper() else "ũ"
        return cluster

    # Apply the Kikuyu specific tilde logic first
    processed_text = re.sub(kikuyu_vowel_pattern, handle_kikuyu_vowels, decomposed_text)

    # 3. Pattern 2: Catch ANY remaining stray combining diacritics on other letters
    # Because the text is decomposed, we can literally just strip out the diacritic symbols
    # from any other consonants or standard vowels (a, e, o).
    stray_diacritics_pattern = r"[\u0300-\u036f]+"
    processed_text = re.sub(stray_diacritics_pattern, "", processed_text)

    # 4. Recompose back to standard NFC format
    return unicodedata.normalize("NFC", processed_text)


def clean_dictionary_typos(text):

    # remove recurring id marker
    text = re.sub(r"\s864403\s[A-Z]", "", text)
    # handle hyphen breaks
    text = re.sub(r"-\s([a-zũĩ])", r"\1", text)

    # 1. Standard static replacements
    text = text.replace("fħ", "ffi")
    text = text.replace("œ", "~").replace("æ", "~")
    text = text.replace("ø", "g")
    text = text.replace("ı", "i")
    text = text.replace("Ħ", "")
    text = text.replace("864403", "")

    # 2. Handle 'ł' variations
    text = text.replace("łi", "i").replace("ił", "i")
    text = text.replace("ł-", "i-")

    # 3. Contextual 'ð' replacement
    # Pattern A: ð surrounded by vowels (case-insensitive) -> delete it
    # [aeiouĩũ] captures standard vowels + the Kikuyu tilde vowels
    vowels = r"[aeiouĩũAEIOUĨŨ]"
    vowel_surrounded_pattern = f"(?<={vowels})ð(?={vowels})"
    text = re.sub(vowel_surrounded_pattern, "", text)

    # Pattern B: Any remaining ð (not surrounded by vowels) -> replace with ũ
    text = text.replace("ð", "ũ")

    return text


def process_dictionary_file(input_filepath, output_filepath):
    """Reads the entire text file, cleans all diacritics, and writes it back."""
    print(f"Reading {input_filepath}...")
    with open(input_filepath, "r", encoding="utf-8") as f:
        content = f.read()

    print(
        "Processing file: \nconverting i/u clusters and stripping other diacritics..."
    )
    cleaned_content = fix_dictionary_diacritics(content)

    print("Applying typo replacement rules...")
    cleaned_content = clean_dictionary_typos(cleaned_content)

    print(f"Saving cleaned file to {output_filepath}...")
    with open(output_filepath, "w", encoding="utf-8") as f:
        f.write(cleaned_content)
    print("Done!")

In [106]:
INPUT_FILE = "kikuyu_extracted_v2.txt"
OUTPUT_FILE = "kikuyu_extracted_v2_cleaned.txt"

process_dictionary_file(INPUT_FILE, OUTPUT_FILE)

Reading kikuyu_extracted_v2.txt...
Processing file: 
converting i/u clusters and stripping other diacritics...
Applying typo replacement rules...
Saving cleaned file to kikuyu_extracted_v2_cleaned.txt...
Done!


# 3. Manual Tagging

In [41]:
# !pip install pyspellchecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 35.5 MB/s eta 0:00:00m eta 0:00:010:00:01

[notice] A new release of pip is available: 23.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
# POS tags — note the leading space is intentional (avoids false matches mid-word)
pos_tags = [
    " n.",
    " n,",
    " a.",
    " n.c.",
    " abb.",
    " a.c.",
    " adv.",
    " adj.",
    " conj.",
    " dem.",
    " ideo.",
    " int.",
    " interj.",
    " interrog.",
    " n.c,",
    " num.",
    " p.a.",
    " p.c.",
    " part.",
    " poss.",
    " pron.",
    " subj.",
    " v.",
    " v.i.",
    " v.t.",
]

prefix_list = [
    "mũ-",
    "i-",
    "ma-",
    "ki-",
    "tũ-",
    "ũ-",
    "ha-",
    "a-",
    "ĩ-",
    "ri-",
    "rũ-",
    "ci-",
    "kũ-",
    "ka-",
    "mĩ-",
    "mi-",
    "n-",
]

In [4]:
def regex_extract_tags(INPUT_FILE):

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        raw_chunks = [e.strip() for e in f.read().split("\n") if len(e.strip()) > 3]

    processed_entries = []
    current_column = "UNKNOWN"

    for chunk in raw_chunks:
        if chunk.upper().startswith("COLUMN-START"):
            current_column = chunk
        else:
            processed_entries.append((chunk, current_column))

    

    extracted_data = []

    for idx, (text, col) in enumerate(processed_entries):
        full_entry = " ".join(text.split())

        # ── 1. WORD: everything before the first comma ────────────────────────
        word_match = re.match(r"^([^,]+),", full_entry)
        if (
            not word_match
        ):  # only 15 such cases, usually cut-off sentences, not significant
            # print(idx,": \n", full_entry,'\n\n')
            extracted_data.append(
                {
                    "INDEX": idx,
                    "PAGE-COLUMN": col.split("COLUMN-START-")[1],
                    "WORD": "",
                    "PREFIX": "",
                    "NC": "",
                    "POS": "",
                    "DESCRIPTION": "",
                    "FULL_ENTRY": full_entry,
                }
            )
            continue

        word = word_match.group(1).strip()

        # ── 2. AFFIX: between 1st and 2nd comma, MUST contain a hyphen ───────
        affix = None
        # Search scope: up to " n." or " n," if present, else up to first comma/semicolon/colon, else first half
        if " n." in full_entry or " n," in full_entry:
            search_scope = re.split(r" n[.,]", full_entry)[0]
        elif any(e in full_entry for e in [",", ";", ":"]):
            search_scope = re.split(r"[,;:]", full_entry)[0]
        else:
            search_scope = full_entry[: len(full_entry) // 2]
        # Find all prefixes from predefined list that appear in search scope,
        # only when surrounded by non-alphabet characters or brackets
        affix_matches = []
        for p in prefix_list:
            p_stripped = p.strip()
            # Build pattern: non-alpha or bracket before, non-alpha or bracket after
            pattern = rf"(?:^|[^\w]|\()\s*({re.escape(p_stripped)})(?:[^\w]|\))"
            if re.search(pattern, search_scope):
                affix_matches.append(p_stripped)
        if affix_matches:
            affix = ", ".join(affix_matches)

        # ── 3. NC: after "cl. " → grab first match, only if before description ────
        nc = None
        nc_match = re.search(
            r"cl\.\s*([\d/a-z]+(?:\s*(?:or|and|only)\s*[\d/a-z]+)*)", full_entry
        )
        if nc_match:
            candidate_nc = nc_match.group(1).strip()
            desc_anchor_pos = full_entry.find("), ")
            nc_pos = nc_match.start()
            if desc_anchor_pos == -1 or nc_pos < desc_anchor_pos:
                nc = candidate_nc

        # ── 4. POS: leading-space anchored to avoid mid-word hits ─────────────
        # Escapes each tag and joins into one alternation; matches the last
        # occurrence so it isn't confused with abbreviations inside descriptions.
        pos = None
        pos_pattern = "|".join(
            re.escape(t) for t in sorted(pos_tags, key=len, reverse=True)
        )
        pos_match = re.search(rf"({pos_pattern})", full_entry)
        if pos_match:
            pos = pos_match.group(1).strip()

        # ── 5. DESCRIPTION: after "), " → nested if/else on terminator ───────────
        description = None
        desc_match = re.search(r"\),\s+(.+)", full_entry, re.DOTALL)
        if desc_match:
            raw = desc_match.group(1).strip()
            if ";" in raw or ":" in raw:
                # Rule 1: stop at first ";" or ":"
                term = re.search(r"(.+?)(?:[;:])", raw, re.DOTALL)
                if term:
                    description = term.group(1).strip()
            else:
                # Rule 2: stop at full-stop
                term = re.search(r"(.+?\.)(?:\s|$)", raw, re.DOTALL)
                if term:
                    description = term.group(1).strip()
            # Rule 3: neither found → None (description stays None)

        extracted_data.append(
            {
                "INDEX": idx,
                "PAGE-COLUMN": col.split("COLUMN-START-")[1],
                "WORD": word,
                "PREFIX": affix,
                "NC": nc,
                "POS": pos,
                "DESCRIPTION": description,
                "FULL_ENTRY": full_entry,
            }
        )

    return pd.DataFrame(extracted_data)

In [5]:
INPUT_FILE = "kikuyu_extracted_v2_cleaned.txt"

kikuyu_tagged_df = regex_extract_tags(INPUT_FILE)

In [6]:
kikuyu_tagged_df.to_csv('kikuyu_complete_cleaned_v2.csv',index=False)

In [7]:
kikuyu_tagged_df.loc[kikuyu_tagged_df["INDEX"] == 4291]

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,FULL_ENTRY
4291,4291,P250-C1,kwini,ma-,1/6 or 1/2,n.,queen.,"kwini, ma-, n. cl. 1/6 or 1/2 (1x), queen. [E...."


In [86]:
# search_str = r"\),\s"
search_str = r"cl\.\s*(\d[\w\s]*?)(?:,|\(|\.)"

nc_kikuyu_df = kikuyu_tagged_df[kikuyu_tagged_df["FULL_ENTRY"].str.contains(search_str)]

# nc_kikuyu_df['FULL_ENTRY'].tolist()

/var/folders/lc/yr0c0sq938g9gtrdrvgrk6nm0000gn/T/ipykernel_91345/1294994596.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kikuyu_tagged_df['FULL_ENTRY'].str.contains(search_str)


In [67]:
# nc_kikuyu_df.to_csv('kikuyu_nc_check.csv',index=False)

# 4. Cleanup

In [83]:
kikuyu_tagged_df_v1 = pd.read_csv("kikuyu_complete_cleaned.csv")

In [65]:
# switch to v2 -- De-skewed
kikuyu_tagged_df = pd.read_csv("kikuyu_complete_cleaned_v2.csv")

In [84]:
def fix_word_column(df):
    def fix_word(word):
        # Rule 1: trailing "!" → append (1), remove "!"
        # Rule 2: trailing "?" → append (2), remove "?"
        # Rule 3: trailing digit → append (digit in parens), remove digit
        # All rules preserve anything after the word (e.g. bracketed alternates)

        # Split word from any trailing bracketed content e.g. " [mũkiyo]"
        if pd.isna(word): return ''
        bracket_match = re.search(r"(\s*\[.*\].*)$", word)
        tail = bracket_match.group(1) if bracket_match else ""
        core = word[: bracket_match.start()] if bracket_match else word

        if ("!" in core) or ("'" in core):
            core = core[:-1].rstrip() + " (1)"
        elif "?" in core:
            core = core[:-1].rstrip() + " (2)"
        elif re.search(r"\d$", core):
            digit = re.search(r"(\d+)$", core).group(1)
            core = re.sub(r"\d+$", "", core).rstrip() + f" ({digit})"

        return core + tail

    df["WORD"] = df["WORD"].apply(fix_word)
    return df


kikuyu_tagged_df = fix_word_column(kikuyu_tagged_df)

In [85]:
# len(kikuyu_tagged_df_v1['WORD'].index), len(kikuyu_tagged_df['WORD'].index)
kikuyu_tagged_df_v1['NC'].count(),kikuyu_tagged_df['NC'].count()

(np.int64(803), np.int64(799))

In [86]:
kikuyu_tagged_df.to_csv("kikuyu_complete_cleaned_v2.csv", index=False)

### Clean NC tags

In [87]:
# strategy: make a gold standard label set of all possible NC annotations??
# 1. Filter valid rows
subset = kikuyu_tagged_df[
    kikuyu_tagged_df["NC"].notna() & (kikuyu_tagged_df["NC"] != "")
]

# 2. Calculate frequencies and sort by most frequent
counts = subset["NC"].value_counts()

# 3. Get one sample row for each noun class
# We take the first appearance and join the frequency count
result = subset.drop_duplicates(subset="NC", keep="first").copy()
result["FREQUENCY"] = result["NC"].map(counts)

# 4. Sort by frequency (descending) and select columns
result = result.sort_values(by="FREQUENCY", ascending=False)
final = result[["INDEX", "WORD", "NC", "PAGE-COLUMN", "FREQUENCY"]]

# 5. Export
# final.to_csv("kikuyu_NC_freq_subset.csv", index=False)

In [70]:
# lookup any NC
# subset[subset["NC"] == "r14"][["NC", "PAGE-COLUMN", "FULL_ENTRY"]].head(30)

,NC,PAGE-COLUMN,FULL_ENTRY


In [88]:
# CLEAN THE ENTIRE DICT using mapping !

# 1. Load the mapping file
mapping_df = pd.read_csv("output_kikuyu_benson_1964/kikuyu_NC_freq_subset_mapped.csv")

# 2. Filter out rows where NOUN_CLASS_ACTUAL is empty/NaN
valid_map = mapping_df.dropna(subset=["NC_FIXED"])
valid_map = valid_map[valid_map["NC_FIXED"].str.strip() != ""]

# 3. Create the dictionary from valid pairs only
mapping_dict = dict(zip(valid_map["NC"], valid_map["NC_FIXED"]))

# 4. Apply mapping: if key not in dict, .map() returns NaN, then .fillna() restores original
kikuyu_tagged_df["NC"] = (
    kikuyu_tagged_df["NC"].map(mapping_dict).fillna(kikuyu_tagged_df["NC"])
)

In [89]:
# get a list of ALL POSSIBLE NC TAGS, without separating into individual - just to verify the cleanup works.

# 1. Force everything to string and strip whitespace to ensure '14' == 14
effective_labels = (
    mapping_df["NC_FIXED"].fillna(mapping_df["NC"]).astype(str).str.strip()
)

# 2. Filter out common "empty" string representations
effective_labels = effective_labels[~effective_labels.isin(["nan", "None", ""])]

# leave out all OCR labels with freq == 1
nc_set_gold = set(effective_labels.tolist()[:])

# nc_set_gold

In [90]:
# final save

kikuyu_tagged_df.to_csv("output_kikuyu_benson_1964/kikuyu_tagged_full.csv", index=False)

### Fix empty DESC

In [91]:
kikuyu_tagged_df = pd.read_csv("output_kikuyu_benson_1964/kikuyu_tagged_full.csv")

In [92]:
# fix empty description
# if subset_df is not None: del subset_df
    
subset_df = kikuyu_tagged_df[
    kikuyu_tagged_df["DESCRIPTION"].isna() | (kikuyu_tagged_df["DESCRIPTION"] == "1.")
].copy()

In [93]:
# if full_entry has no pos tag, and has only one full stop, or only two commas, then mark this as "reference" and move on.
# for the others (filter by not "reference"), extract a description.

In [94]:
ref_list_df = subset_df[
    (
        (subset_df["POS"].isna())
        & (subset_df["FULL_ENTRY"].notna())
        & ((subset_df["FULL_ENTRY"].str.count("\\.") == 1))
        | (subset_df["FULL_ENTRY"].str.contains(", see"))
    )
]

ref_list = ref_list_df["FULL_ENTRY"].tolist()

In [95]:
len(ref_list)

2

In [79]:
len(
    [e for e in ref_list if ", see" in e]
)  # there were only 17 real entries that we miss among the 200, so letting this condition be the OR check.

2

In [81]:
ref_list_df.loc[:, "DESCRIPTION"] = "REFERENCE"

In [82]:
ref_list_df

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,DESCRIPTION_SPELLCHECK,FULL_ENTRY
4396,4396,P254-C1,mathaka (N.K.),NaN,NaN,NaN,REFERENCE,"see thamaka,.","mathaka (N.K.), see thamaka,."
9813,9813,P539-C2,mũ (N.K.),NaN,NaN,NaN,REFERENCE,"mũ (N.K.), see mũhu,.","mũ (N.K.), see mũhu,."


In [42]:
subset_df.update(
    ref_list_df, overwrite=True
)  # overwrites all new entries excluding NaNs

In [43]:
entries = subset_df[subset_df["DESCRIPTION"] != "REFERENCE"]["FULL_ENTRY"].tolist()

In [42]:
GIKUYU_CHARS = re.compile(r"[ĩũʻ]")

abbr_list = pos_tags + [
    'expr.', 'ref.','N.K.','I. ','descr.','arch.','r.','x.','indic.','xĩ.','cf.','Bibl.','sp.',
    'dim.','sth.', 'poss.','sth.', 'esp.', 'sb.', 'usu.','infin.','imperat.','indic.',
    'part.','interj.','expr.','c.', 'Ind.', 'cf.', 'dim.', 'caus.', 'rec.',
    'app.', 'inten.', 'diff.', 'assoc.', 'pot.', 'st.', 'an.','cl.','coll.','e.g.',':','N-K.','N.K,','1.',
    'a.'
]

def is_gikuyu_lead_clause(clause):
    """True if this leading clause is a Gikuyu example phrase/word, not English gloss text."""
    if not clause:
        return True
    if clause.startswith("~"):
        return True
    if GIKUYU_CHARS.search(clause):
        return True
    if re.match(r"^-?[a-z']{1,4}\s*~$", clause):
        return True
    if re.match(
        r"^\(used\s", clause, re.IGNORECASE
    ):  # "(used adv.)" style parenthetical lead
        return True
    if re.match(r"^(see|under)\b", clause, re.IGNORECASE):
        return True
    return False


def find_class_zone_end(entry):
    """Return index right after the noun-class apparatus ends. This extraction is focused on nouns.
    The content INSIDE (...) is never meaningful for the description (it's a class
    code, possibly OCR-garbled like (xX), (V11), (vin), (xI-IH1)} -- so we just need
    to find where that bracketed zone closes, not validate what's inside it."""
    # explicit "cl. ... ;" form
    m = re.search(r"cl\.[^;]*?(?:\)|,)\s*;?", entry)
    if m:
        end = m.end()
        if end < len(entry) and entry[end] == ";":
            end += 1
        return end
    # ANY parenthetical right after "n." -- don't validate contents, just find the close.
    # Search starting from the "n." marker so we don't grab an earlier unrelated "(...)"
    # e.g. "gĩcinicini [gĩcinicini], i-, n. (xX);" -- skip the leading "[gĩcinicini]" bracket.
    n_marker = re.search(r"\bn\.\s*", entry)
    search_from = n_marker.end() if n_marker else 0
    m = re.search(r"\(([^()]*)\)[\}\)]?", entry[search_from:])
    if m:
        return search_from + m.end()
    # bare class code with no parens at all, e.g. "n. TII, 1. ..."
    # handled by the "1." anchor below, so just fall back to after "n."
    if n_marker:
        return n_marker.end()
    return 0

def remove_abbreviations(entry, abbr_list):
    sorted_abbr = sorted(abbr_list, key=len, reverse=True)
    pattern = "|".join(re.escape(a) for a in sorted_abbr)
    cleaned = re.sub(rf"\b(?:{pattern})\.*", "", entry)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()
    return cleaned

def extract_first_sense(entry):
    start = find_class_zone_end(entry)
    rest = entry[start:].strip(" ,;")

    # Keep a raw copy with abbreviations intact for final description cutting
    original_rest = rest  

    rest = remove_abbreviations(' ' + rest, abbr_list)

    m1 = re.search(r"1\.\s*", rest)
    if m1:
        rest = rest[m1.end() :]
    else:
        rest = re.sub(r"^1\.\s*", "", rest)

    while True:
        m = re.match(r"^([^,;.]+)[,;]\s*", rest)
        if m and is_gikuyu_lead_clause(m.group(1).strip()):
            rest = rest[m.end() :]
            continue
        break

    rest = re.sub(
        r"^\((?:used|usu\.?|esp\.?|chiefly|mostly)\b[^()]*\)\s*",
        "",
        rest,
        flags=re.IGNORECASE,
    )

    # --- FIXED TERMINATION LOGIC ---
    # Scan characters one by one to find the true sentence terminator
    search_idx = 0
    while search_idx < len(rest):
        # Find the next potential terminator
        m_term = re.search(r"[.;]", rest[search_idx:])
        if not m_term:
            # No more terminators found, take everything left
            search_idx = len(rest)
            break
            
        term_pos = search_idx + m_term.start()
        
        if rest[term_pos] == ';':
            # Semicolons are never part of abbreviations; stop here safely
            search_idx = term_pos
            break
            
        if rest[term_pos] == '.':
            # Check if this period is part of an abbreviation
            matched_abbr = None
            # Look backwards from the period to see if an abbreviation matches
            for abbr in sorted(abbr_list, key=len, reverse=True):
                # If the abbreviation ends with a period, locate where it would start
                if abbr.endswith('.'):
                    start_look = term_pos - len(abbr) + 1
                    if start_look >= 0 and rest[start_look:term_pos + 1].lower() == abbr.lower():
                        matched_abbr = abbr
                        break
            
            if matched_abbr:
                # Skip past the remaining expected periods of this abbreviation
                # e.g., if N.K. is found, skip past its length so we don't trip on its internal periods
                search_idx = start_look + len(matched_abbr)
            else:
                # It's a real terminal full stop!
                search_idx = term_pos
                break

    desc = rest[:search_idx]
    return desc.strip(" ,;.")

# print(f"{len(entries)} entries\n")
# for e in entries:
#     print(repr(e[:100]), "->\n", repr(extract_first_sense(e)), "\n\n")

In [7]:
# 1. Compute the extracted senses across the entire subset_df
extracted_series = subset_df[
    (subset_df["DESCRIPTION"] != "REFERENCE") & 
    (subset_df["FULL_ENTRY"].notna())
]["FULL_ENTRY"].apply(extract_first_sense)

# 2. TARGET LOGIC: Select every index in subset_df EXCEPT the ones in ref_list_df
ref_indices = ref_list_df["INDEX"].tolist()
target_indices = subset_df.index.difference(ref_indices)

# 3. Intersect targets with your extracted rows to avoid any mapping gaps
valid_targets = extracted_series.index.intersection(target_indices)

# 4. Safely update DESCRIPTION directly using .loc
subset_df.loc[valid_targets, "DESCRIPTION"] = extracted_series.loc[valid_targets]

NameError: name 'subset_df' is not defined

In [8]:
subset_df.to_csv("subset_df.csv", index=False)

NameError: name 'subset_df' is not defined

In [53]:
# apply new DESC logic to the ENTIRE dataset

# 1. Compute the extracted senses across the entire subset_df
extracted_series = kikuyu_tagged_df[
    (kikuyu_tagged_df["FULL_ENTRY"].notna())
]["FULL_ENTRY"].apply(extract_first_sense)

# 2. TARGET LOGIC: Select every index in subset_df EXCEPT the ones in subset_df
subset_indices = subset_df["INDEX"].tolist()
target_indices = kikuyu_tagged_df.index.difference(subset_indices)

# 3. Intersect targets with your extracted rows to avoid any mapping gaps
valid_targets = extracted_series.index.intersection(target_indices)

# 4. Safely update DESCRIPTION directly using .loc
kikuyu_tagged_df.loc[valid_targets, "DESCRIPTION"] = extracted_series.loc[valid_targets]

NameError: name 'subset_df' is not defined

In [150]:
# finally, merge subset_df into the main df

# 1. Identify the rows in the main DataFrame that exist in the subset
overlapping_indices = kikuyu_tagged_df.index.intersection(subset_df.index)

# 2. Overwrite those exact rows entirely with subset_df's data
kikuyu_tagged_df.loc[overlapping_indices] = subset_df.loc[overlapping_indices]

In [151]:
kikuyu_tagged_df.to_csv("kikuyu_complete_cleaned_v2.csv", index=False)

In [50]:
import pandas as pd

def generate_description_diffs(file1_path, file2_path):
    # 1. Load the CSV files
    # Note: If your CSVs have a specific ID/Index column, add: index_col="YOUR_INDEX_COL"
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)
    
    # 2. Ensure they are aligned on the same shape and indices
    if len(df1) != len(df2):
        print(f"Warning: File row counts differ ({len(df1)} vs {len(df2)}). Diffs will align on row index.")

    # 3. Create a boolean mask where the descriptions do not match
    # (Handling potential NaN/Null values safely using .ne)
    changed_mask = df1['DESCRIPTION'].ne(df2['DESCRIPTION'])
    
    # 4. Filter out the rows that changed
    diff_df = pd.DataFrame({
        'ROW_INDEX': df1[changed_mask].index,
        'ORIGINAL_VALUE': df1.loc[changed_mask, 'DESCRIPTION'],
        'NEW_VALUE': df2.loc[changed_mask, 'DESCRIPTION'],
        'FULL_ENTRY': df1.loc[changed_mask, 'FULL_ENTRY']
    })
    
    # 5. Output results
    total_changes = len(diff_df)
    print(f"=== Comparison Report ===")
    print(f"Total number of row changes: {total_changes}\n")
    
    # Optional: Save the diff report to a new CSV file
    diff_df.to_csv('desc_update_diff.csv', index=False)
    
    return diff_df

# --- Run the function ---
# generate_description_diffs('kikuyu_complete_cleaned.csv', 'kikuyu_complete_cleaned_v2.csv') # comapre logic flawed


# Description v2 -- Spellcheck

In [11]:
# generate HTML diffs between two text files
import difflib
from pathlib import Path

file1_lines = Path('output_kikuyu_benson_1964/kikuyu_extracted_full_cleaned.txt').read_text().splitlines()
file2_lines = Path('kikuyu_extracted_v2_cleaned.txt').read_text().splitlines()

# Set wrapcolumn to break lines after N characters
diff_maker = difflib.HtmlDiff(wrapcolumn=80)

# html_diff = diff_maker.make_file(file1_lines, file2_lines)
# Path('diff_v2.html').write_text(html_diff)   

In [43]:
from spellchecker import SpellChecker
import unicodedata

spell = SpellChecker(language='en')

def strip_known_fields(full_entry, word, affix, nc, pos):
    """
    Remove already-extracted field values from the entry so spellcheck
    doesn't false-trigger on them. Works on a copy — does not mutate the row.
    """
    scrubbed = full_entry
    for field in [word, affix, nc, pos]:
        if field:
            scrubbed = scrubbed.replace(field, " ", 1)
    # Also strip structural markers that aren't English
    for tag in pos_tags + prefix_list + [' cl.', 'see under', ' cf.', ' n.', ' v.']:
        scrubbed = re.sub(rf'(?<!\w){re.escape(tag.strip())}(?!\w)', ' ', scrubbed)
    # Collapse multiple spaces
    scrubbed = re.sub(r'\s{2,}', ' ', scrubbed).strip()
    return scrubbed


def mask_abbreviation_stops(text):
    """
    Replace the full-stop(s) that belong to an abbreviation with § so they
    don't act as sentence terminators downstream. The abbreviation text itself
    is kept intact.
    """
    sorted_abbr = sorted(abbr_list, key=len, reverse=True)
    pattern = "|".join(re.escape(a) for a in sorted_abbr)
    def _mask(m):
        return m.group(0).replace(".", "§")
    return re.sub(rf'(?<!\w)(?:{pattern})\.?', _mask, text)


def unmask_abbreviation_stops(text):
    return text.replace("§", ".")


def find_first_english_index(scrubbed_text, original_text):
    tokens = re.finditer(
        r"\([^)]+\)|[^\W\d_]+(?:[\''\-][^\W\d_]+)*",
        scrubbed_text,
        re.UNICODE
    )

    # Build abbr_list lookup set — strip dots and lowercase for comparison
    abbr_set = {re.sub(r'\.', '', a).strip().lower() for a in abbr_list}

    for token in tokens:
        token_text = token.group(0)
        start_idx  = token.start()
        end_idx    = token.end()

        if (start_idx > 0 and scrubbed_text[start_idx - 1].isalnum()) or \
           (end_idx < len(scrubbed_text) and scrubbed_text[end_idx].isalnum()):
            continue

        if end_idx < len(scrubbed_text) and scrubbed_text[end_idx] == '?':
            continue

        # Reject tokens that BEGIN or END with hyphen (bare affixes),
        # but allow internal hyphens (e.g. "bee-keeper", "well-known")
        if token_text.startswith('-') or token_text.endswith('-'):
            continue

        normalized = "".join(
            c for c in unicodedata.normalize('NFKD', token_text)
            if not unicodedata.combining(c)
        )

        if token_text.startswith('('):
            words = re.findall(r'\b[a-z]+\b', normalized.lower())
        else:
            # For hyphenated tokens, test both the whole token AND its parts.
            # spell.known() won't recognise "bee-keeper" as a unit, but will
            # recognise "bee" and "keeper" individually.
            parts = re.split(r"[-']", normalized.lower())
            words = [normalized.lower()] + [p for p in parts if len(p) > 1]

        words = [w for w in words if len(w) > 1]
        if not words:
            continue

        # Match against abbr_list by stripping dots and lowercasing
        has_abbrev  = any(re.sub(r'\.', '', w).strip().lower() in abbr_set for w in words)
        valid_words = spell.known(words)

        if valid_words or has_abbrev:
            original_pos = original_text.find(token_text)
            return original_pos if original_pos != -1 else None

    return None


def extract_english_description(row):
    if isinstance(row, str):
        full_entry = row
        word, affix, nc, pos = "", "", "", ""
    else:
        full_entry = row.get("FULL_ENTRY", "")
        word       = row.get("WORD", "")
        affix      = row.get("AFFIX", "")
        nc         = row.get("NC", "")
        pos        = row.get("POS", "")

    # Step 1: strip known fields so spellcheck isn't misled by native-language content
    scrubbed = strip_known_fields(full_entry, word, affix, nc, pos)

    # Step 2: mask abbreviation full-stops in the ORIGINAL entry
    # so they don't act as sentence boundaries when we slice
    masked_entry = mask_abbreviation_stops(full_entry)

    # Step 3: find where English starts (using scrubbed text, map back to original)
    start_index = find_first_english_index(scrubbed, masked_entry)
        
    if start_index is None:
        return None

    raw_desc = masked_entry[start_index:].strip()

    # Step 4: slice at structural terminators — priority: ";" or ":", then ".", then None
    if ";" in raw_desc or ":" in raw_desc:
        term_match = re.search(r'^(.+?)(?:[;:])', raw_desc, re.DOTALL)
        description = term_match.group(1).strip() if term_match else None
    else:
        # Stop at a real sentence-ending period (§ placeholders are safe to pass through)
        term_match = re.search(r'^(.+?\.)(?:\s|$)', raw_desc, re.DOTALL)
        description = term_match.group(1).strip() if term_match else None

    # Step 5: restore masked stops in the final description
    return unmask_abbreviation_stops(description) if description else None 

In [96]:
kikuyu_tagged_df = pd.read_csv("kikuyu_complete_cleaned_v2.csv")

In [97]:
# apply Spellcheck-based description extraction to all entries (except REFERENCE)

# apply new DESC logic to the ENTIRE dataset

# 1. Compute the extracted senses across the entire subset_df
extracted_series = kikuyu_tagged_df[
    (kikuyu_tagged_df["DESCRIPTION"] != "REFERENCE") &
    (kikuyu_tagged_df["FULL_ENTRY"].notna())
]["FULL_ENTRY"].apply(extract_english_description) # SPELLCHECK BASED

# 2. TARGET LOGIC: Select every index in kikuyu_tagged_df EXCEPT the ones to BE SKIPPED
skip_indices = kikuyu_tagged_df[(kikuyu_tagged_df["DESCRIPTION"] == "REFERENCE")].index
target_indices = kikuyu_tagged_df.index.difference(skip_indices)

# 3. Intersect targets with your extracted rows to avoid any mapping gaps
valid_targets = extracted_series.index.intersection(target_indices)

# 4. Safely update DESCRIPTION directly using .loc
kikuyu_tagged_df.loc[valid_targets, "DESCRIPTION_SPELLCHECK"] = extracted_series.loc[valid_targets]

kikuyu_tagged_df = kikuyu_tagged_df.loc[:,[
    'INDEX','PAGE-COLUMN','WORD','PREFIX','NC','POS','DESCRIPTION','DESCRIPTION_SPELLCHECK','FULL_ENTRY'
]
    ]

In [98]:
kikuyu_tagged_df.to_csv("kikuyu_complete_cleaned_v2.csv", index=False)

In [47]:
kikuyu_tagged_df[
    kikuyu_tagged_df['DESCRIPTION'].str[0:3] != kikuyu_tagged_df['DESCRIPTION_SPELLCHECK'].str[0:3]
].to_csv('kikuyu_v2_diff.csv',index=False)

## v1 vs. v2

In [121]:
def select_best_description(row):
    full_entry = row.get("FULL_ENTRY", "")
    v1         = row.get("DESCRIPTION", "")
    v2         = row.get("DESCRIPTION_SPELLCHECK", "")
    pos        = row.get("POS", "")
    
    # 1. check pos and v2 for None, NaN, or raw float objects
    if (pd.isna(pos) or pos is None) or (pd.isna(v2) or v2 is None): return v1

    # ── Rule 4b applied early: clean ")," preamble from both before any comparison ──
    def clean_paren(text):
        if not text:
            return text
        paren_match = re.search(r"\),\s*", text)
        return text[paren_match.end():].strip() if paren_match else text

    # ── Rule 4a: if " cl. " or " cl " found, drop everything up to next comma/semicolon ──
    def clean_cl(text):
        if not text:
            return text
        cl_match = re.search(r" cl[. ].+?[,;]\s*", text)
        return text[cl_match.end():].strip() if cl_match else text

    def apply_cleaning(text):
        text = clean_paren(text)
        text = clean_cl(text)
        return text.strip() if text else text

    # ── Clip long descriptions at first period, semicolon, or colon ──────────
    def clip_if_long(text):
        if not text or len(text) <= 100:
            return text
        # Build pattern that matches abbreviation full-stops to skip over
        sorted_abbr = sorted(abbr_list, key=len, reverse=True)
        abbr_pattern = "|".join(re.escape(a) for a in sorted_abbr)
        # Scan character by character via finditer, skipping abbr matches
        # First, mask abbreviation periods so they don't look like terminators
        masked = re.sub(rf'(?:{abbr_pattern})', lambda m: m.group(0).replace(".", "§"), text)
        # Now find the first real terminator in the masked string
        clip_match = re.search(r"[.;:]", masked)
        if clip_match:
            return text[:clip_match.end()].strip()
        return text

    v1 = apply_cleaning(v1)
    v2 = apply_cleaning(v2)

    # ── Guard: if both empty after cleaning, return None ─────────────────────
    if not v1 and not v2:
        return None

    # ── Rule 1: if first 3 chars match, prefer v1 ────────────────────────────
    if v1 and v2 and v1[:3] == v2[:3]:
        best = v1
    else:
        # ── Rule 2: determine start marker position ───────────────────────────
        start_marker = None

        pos_pos = full_entry.find(" " + pos) if pos else -1

        # Look for " cl." or " cl " after POS tag — if found, skip past next comma/semicolon
        if pos_pos != -1:
            after_pos = full_entry[pos_pos + len(pos):]
            cl_after_pos = re.match(r"\s*cl[. ].+?[,;]\s*", after_pos)
            if cl_after_pos:
                start_marker = pos_pos + len(pos) + cl_after_pos.end()

        # Look for ")," after the POS tag within 15 characters (if cl. not found)
        if start_marker is None and pos_pos != -1:
            window = full_entry[pos_pos: pos_pos + len(pos) + 15]
            paren_match = re.search(r"\),", window)
            if paren_match:
                start_marker = pos_pos + paren_match.end()

        # Fallback: place start marker right after " {POS}"
        if start_marker is None and pos_pos != -1:
            start_marker = pos_pos + len(pos)

        # ── Rule 3: pick whichever description starts closer to start_marker ──
        if start_marker is not None:
            v1_pos = full_entry.find(v1[:10]) if v1 else -1
            v2_pos = full_entry.find(v2[:10]) if v2 else -1

            v1_dist = abs(v1_pos - start_marker) if v1_pos != -1 else float('inf')
            v2_dist = abs(v2_pos - start_marker) if v2_pos != -1 else float('inf')

            if v2_pos == -1 or v1_dist <= v2_dist:
                best = v1
            else:
                best = v2
        else:
            best = v1 if v1 else v2

    # ── Safety net: if best is empty, fall back to v1, then v2 ───────────────
    if not best or len(best) == 0:
        best = apply_cleaning(row.get("DESCRIPTION", ""))
    if not best or len(best) == 0:
        best = apply_cleaning(row.get("DESCRIPTION_SPELLCHECK", ""))

    # ── Clip if over 100 characters ───────────────────────────────────────────
    best = clip_if_long(best)

    if best == '1.': return v1

    return best.strip() if best else None

In [122]:
kikuyu_tagged_df["DESCRIPTION_WINNER"] = kikuyu_tagged_df[
    (kikuyu_tagged_df["FULL_ENTRY"].notna()) &
    (kikuyu_tagged_df["DESCRIPTION"].notna())
].apply(select_best_description, axis=1)

In [123]:
kikuyu_tagged_df = kikuyu_tagged_df.loc[:,[
    'INDEX','PAGE-COLUMN','WORD','PREFIX','NC','POS','DESCRIPTION','DESCRIPTION_SPELLCHECK','DESCRIPTION_WINNER','FULL_ENTRY'
]
    ]

In [124]:
kikuyu_tagged_df.to_csv("kikuyu_complete_cleaned_v2_winner.csv", index=False)

In [125]:
kikuyu_tagged_df[
    kikuyu_tagged_df['DESCRIPTION'].str[0:3] != kikuyu_tagged_df['DESCRIPTION_SPELLCHECK'].str[0:3]
].to_csv('kikuyu_v2_diff.csv',index=False)

In [127]:
kikuyu_tagged_df[
    (kikuyu_tagged_df['DESCRIPTION_WINNER'] == '1.')
]

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,DESCRIPTION_SPELLCHECK,DESCRIPTION_WINNER,FULL_ENTRY
